# 🏋️‍♂️ Human–Object Interaction (HOI) Multimodal BiLSTM for Sports Action Recognition
## 💻 Local Machine Interactive Notebook (VS Code / Local GPU)

**Thesis Framework**: *Human–Object Interaction-Aware Multimodal CNN–LSTM Framework for Sports Action Recognition*

### 📋 Local Configuration Details:
- **Root Workspace**: `E:\Akhi Thesis`
- **Dataset Folder**: `Model Data` (`Train` 70% = 6,408 videos | `Test` 30% = 2,743 videos)
- **Compute Device**: Local GPU (**NVIDIA GeForce GTX 1650 with Max-Q Design** on CUDA 12.1)
- **Output Artifacts**: Checkpoints in `checkpoints/M7/` and 5 thesis diagrams in `diagrams/M7/`

## 1. Verify Local GPU and Hardware Acceleration

In [ ]:
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Local GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Verify `Model Data` Labeled Train/Test Split

In [ ]:
from pathlib import Path

train_dir = Path("Model Data/Train")
test_dir = Path("Model Data/Test")

train_videos = list(train_dir.rglob("*.avi"))
test_videos = list(test_dir.rglob("*.avi"))
train_classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
test_classes = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])

print(f"Train Categories: {len(train_classes)} | Videos: {len(train_videos):,}")
print(f"Test Categories : {len(test_classes)} | Videos: {len(test_videos):,}")
print(f"Total Videos    : {len(train_videos) + len(test_videos):,}")

# Verify zero overlap
overlap = set(p.name for p in train_videos).intersection(set(p.name for p in test_videos))
print(f"Overlap count   : {len(overlap)} (Zero overlap verified!)")

## 3. Multimodal Feature Preprocessing & Extraction
Extracts Visual (2048-d), Pose (85-d), Object (7-d), and HOI (16-d) features on local GPU.

- **Option A (2-minute smoke test)**: `!python run_local.py --extract --max-per-class 2`
- **Option B (Full dataset extraction)**: `!python run_local.py --extract`

In [ ]:
# Run extraction (toggle --max-per-class 2 for quick testing)
!python run_local.py --extract --data-dir "Model Data" --features-dir "data/features"

# Verify feature tensor readiness
!python run_local.py --verify --features-dir "data/features"

## 4. Train HOI-BiLSTM Model (Mode M7)
Trains 2-Layer BiLSTM with Temporal Attention on local GPU.

In [ ]:
!python run_local.py --train --features-dir "data/features" --model-mode M7 --epochs 40 --batch-size 32 --lr 0.0005

## 5. Render Outcome Diagrams in VS Code

In [ ]:
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("diagrams/M7")
diagrams = [
    ("Diagram 1: Loss Learning Curves", diag_dir / "loss_curves.png"),
    ("Diagram 2: Top-1 & Top-5 Accuracy Curves", diag_dir / "accuracy_curves.png"),
    ("Diagram 3: 82-Class Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("Diagram 4: Per-Class Accuracy Rankings", diag_dir / "per_class_accuracy.png"),
    ("Diagram 5: Temporal Attention Weights Heatmap", diag_dir / "temporal_attention_weights.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        display(Image(filename=str(img_path), width=800))
    else:
        print(f"[!] Diagram not found: {img_path}")

## 8. Real-Time Action Validation & Live Camera Web App
Launches an interactive web app with a public share link (`https://xxxx.gradio.live`).
- Upload any custom sports video (.mp4, .avi, .mov)
- Record live from your webcam directly in the browser
- Real-time visualization of 17 skeletal joints, sports equipment boxes, and HOI vectors
- Live Top-5 prediction confidence chart

In [ ]:
# Launch Gradio Web App with public shareable link
!python app_web.py --share